# SwinUnet Encoder-Only Fine-Tuning & Test (Colab)

This notebook fine-tunes SwinUnet on the WildfireSpreadTS HDF5 dataset and evaluates on the held-out test split, reporting Average Precision (AP), F1, Precision, and Recall.

**Transfer mode:**
- Loads **encoder-only** weights from a branch-specific pre-training checkpoint
- Skips decoder / reconstruction head weights from the pre-training task

**Modes:**
- **Single fold** (`RUN_ALL_FOLDS = False`): trains and tests one fold for quick iteration
- **Full 12-fold** (`RUN_ALL_FOLDS = True`): trains all 12 folds from scratch, reports mean +/- std

**Configurable:**
- `USE_FACTORED_EMBED`: toggle between FactoredPatchEmbed (temporal mixer) and original channel-stacked PatchEmbed
- `BASE_LR`, `MAX_EPOCHS`, `BATCH_SIZE`, `N_LEADING_OBSERVATIONS`, etc.

**Prerequisites:**
- HDF5 dataset already on Google Drive (from `download_and_convert_dataset.ipynb`)
- **Runtime -> Change runtime type -> GPU or TPU** (T4/V100 GPU or TPU v2/v3 recommended)
- A [GitHub Personal Access Token](https://github.com/settings/tokens) stored as a Colab secret named `GITHUB_TOKEN` (Colab sidebar -> key icon -> Add new secret)


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration (user-editable)

In [ ]:
REPO_ORG   = "amindell11"   # Replace with your GitHub username or organisation
REPO_NAME  = "wildfire-ts-swin"
REPO_BRANCH = "feature/simim-pretrain"
HDF5_DIR   = "/content/drive/MyDrive/wildfire_dataset/hdf5"
OUTPUT_BASE = f"/content/drive/MyDrive/wildfire_runs_simim_pretrain_encoder_only"

DATA_FOLD_ID             = 0      # 0-11; which train/val/test year split (ignored when RUN_ALL_FOLDS=True)
RUN_ALL_FOLDS            = False  # True = train & test all 12 folds, report mean +/- std
N_LEADING_OBSERVATIONS   = 1      # 1 or 5
USE_FACTORED_EMBED       = True   # True = FactoredPatchEmbed (temporal mixer), False = channel-stacked
MAX_EPOCHS               = 100
BATCH_SIZE               = 16
BASE_LR                  = 1e-4
FOCAL_GAMMA              = 2.0
CROP_SIDE_LENGTH         = 128
SEED                     = 42
NUM_WORKERS              = 4      # parallel data workers; 4 is safe on Colab

# Pre-trained weights from this branch's pre-training pipeline.
PRETRAIN_WEIGHTS = "/content/drive/MyDrive/wildfire_runs/pretrain/pretrain_best.pth"
PRETRAIN_LOAD_MODE = "encoder_only"  # "encoder_only" or "all_compatible"

# Checkpointing
CHECKPOINT_INTERVAL = 10          # save a full training state every N epochs
RESUME_CHECKPOINT   = None        # path to resume from (single-fold mode only)

# Derived -- single-fold output dir for backwards compat
OUTPUT_DIR = f"{OUTPUT_BASE}/fold{DATA_FOLD_ID}"

## 2b. Copy HDF5 files to local disk

Google Drive I/O is ~100Ã— slower than local NVMe. Copying once here keeps the A100 busy instead of waiting on Drive reads. Takes 1â€“3 min depending on dataset size; skips files already copied.

In [ ]:
import os, shutil, time
from concurrent.futures import ThreadPoolExecutor, as_completed

_LOCAL_HDF5 = "/content/hdf5"
_COPY_WORKERS = 8  # parallel Driveâ†’local transfers

# Collect all (src, dst) pairs
_pairs = []
for _root, _dirs, _files in os.walk(HDF5_DIR):
    _rel = os.path.relpath(_root, HDF5_DIR)
    _dst_dir = os.path.join(_LOCAL_HDF5, _rel)
    os.makedirs(_dst_dir, exist_ok=True)
    for _f in _files:
        _src = os.path.join(_root, _f)
        _dst = os.path.join(_dst_dir, _f)
        if not os.path.exists(_dst):
            _pairs.append((_src, _dst))

if not _pairs:
    print(f"All files already present at {_LOCAL_HDF5}")
else:
    print(f"Copying {len(_pairs)} files with {_COPY_WORKERS} parallel workers...")
    _t0 = time.time()
    _done = 0
    with ThreadPoolExecutor(max_workers=_COPY_WORKERS) as _pool:
        _futs = {_pool.submit(shutil.copy2, s, d): (s, d) for s, d in _pairs}
        for _fut in as_completed(_futs):
            _fut.result()  # re-raises any exception
            _done += 1
            if _done % 20 == 0 or _done == len(_pairs):
                print(f"  {_done}/{len(_pairs)} files copied  ({time.time()-_t0:.0f}s elapsed)")
    print(f"Done in {time.time() - _t0:.0f}s")

HDF5_DIR = _LOCAL_HDF5  # all subsequent cells read from local disk
print(f"HDF5_DIR â†’ {HDF5_DIR}")

## 2c. (One-time) Clone Branch To Google Drive

This stores a **branch-specific** copy on Drive so it does not conflict with other experiment branches.  
After it finishes you can open this notebook directly from Drive and skip the `/content` clone in the next section.


In [ ]:
import os, subprocess

BRANCH_SLUG = REPO_BRANCH.split('/')[-1].replace('-', '_')
DRIVE_REPO_DIR = f"/content/drive/MyDrive/{REPO_NAME}_{BRANCH_SLUG}"
REPO_URL = f"https://github.com/{REPO_ORG}/{REPO_NAME}.git"

if os.path.isdir(os.path.join(DRIVE_REPO_DIR, ".git")):
    print(f"Branch repo already on Drive at {DRIVE_REPO_DIR} -- pulling {REPO_BRANCH} ...")
    subprocess.run(["git", "-C", DRIVE_REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", DRIVE_REPO_DIR, "checkout", REPO_BRANCH], check=True)
    r = subprocess.run(["git", "-C", DRIVE_REPO_DIR, "pull", "origin", REPO_BRANCH], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print(f"Cloning branch {REPO_BRANCH} into {DRIVE_REPO_DIR} ...")
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, DRIVE_REPO_DIR],
        check=True,
    )
    print("Clone complete.")

print(f"
Once you open this notebook from Drive, replace the sys.path cell with:")
print(f"  sys.path.insert(0, '{DRIVE_REPO_DIR}')")


## 3. Clone repo and install dependencies

In [ ]:
from google.colab import userdata
_repo_url = f"https://github.com/{REPO_ORG}/{REPO_NAME}.git"
!rm -rf /content/wildfire-ts-swin
!git clone --branch $REPO_BRANCH $_repo_url /content/wildfire-ts-swin
!pip install -q -r /content/wildfire-ts-swin/requirements.txt
!git -C /content/wildfire-ts-swin rev-parse --abbrev-ref HEAD
!git -C /content/wildfire-ts-swin log --oneline -n 5

In [ ]:
import sys
sys.path.insert(0, f'/content/{REPO_NAME}')

## 4. Detect accelerator (GPU / TPU / CPU)

In [ ]:
!nvidia-smi

In [ ]:
import torch

# Detect the best available device: TPU (XLA) > GPU (CUDA) > CPU
try:
    import torch_xla.core.xla_model as xm
    DEVICE = xm.xla_device()
    print(f"TPU device detected: {DEVICE}")
except ImportError:
    if torch.cuda.is_available():
        DEVICE = torch.device('cuda')
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        DEVICE = torch.device('cpu')
        print("WARNING: No GPU or TPU detected â€” running on CPU (will be slow).")

In [ ]:
## 5a. Launch TensorBoard (optional — run before training; refresh the board while training runs)
import os
# Point TensorBoard at the base dir so it picks up all folds
_tb_logdir = OUTPUT_BASE if RUN_ALL_FOLDS else f"{OUTPUT_DIR}/log"
os.makedirs(_tb_logdir, exist_ok=True)
%load_ext tensorboard
%tensorboard --logdir {_tb_logdir}

## 5. Train & Evaluate

When `RUN_ALL_FOLDS = False`, trains and evaluates a single fold (`DATA_FOLD_ID`).  
When `RUN_ALL_FOLDS = True`, loops over all 12 folds, trains each from scratch, evaluates on the held-out test split, and reports mean ± std AP at the end.

In [ ]:
import os
import random
import types
import numpy as np
import torch
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from config import get_config
from networks.vision_transformer import SwinUnet
from trainer_wildfire import trainer_wildfire
from datasets.wildfire import WildfireDataset, N_FEATURES_PER_TIMESTEP, get_year_split
from utils import compute_binary_metrics, compute_ap

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

ENCODER_STATE_PREFIXES = (
    'patch_embed.',
    'absolute_pos_embed',
    'layers.',
    'norm.',
)


def _make_args(fold_id, output_dir, resume=None):
    """Build the args namespace for a single fold."""
    in_chans = N_FEATURES_PER_TIMESTEP
    extra_opts = [
        'MODEL.SWIN.IN_CHANS', str(in_chans),
        'MODEL.SWIN.N_TIMESTEPS', str(N_LEADING_OBSERVATIONS),
        'MODEL.PRETRAIN_CKPT', 'None',
    ]
    return types.SimpleNamespace(
        data_dir=HDF5_DIR,
        output_dir=output_dir,
        n_leading_observations=N_LEADING_OBSERVATIONS,
        n_leading_observations_test_adjustment=N_LEADING_OBSERVATIONS,
        crop_side_length=CROP_SIDE_LENGTH,
        load_from_hdf5=True,
        data_fold_id=fold_id,
        max_epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        base_lr=BASE_LR,
        num_workers=NUM_WORKERS,
        eval_interval=1,
        seed=SEED,
        n_gpu=1,
        focal_gamma=FOCAL_GAMMA,
        use_factored_embed=USE_FACTORED_EMBED,
        cfg=f'/content/{REPO_NAME}/configs/swin_tiny_patch4_window4_128_wildfire.yaml',
        opts=extra_opts,
        zip=False,
        cache_mode='part',
        resume=resume,
        checkpoint_interval=CHECKPOINT_INTERVAL,
        accumulation_steps=None,
        use_checkpoint=False,
        amp_opt_level='O1',
        tag=None,
        eval=False,
        throughput=False,
    )


def _normalize_pretrained_state_dict(pretrained):
    if isinstance(pretrained, dict) and 'model_state' in pretrained:
        pretrained = pretrained['model_state']
    if isinstance(pretrained, dict) and 'state_dict' in pretrained:
        pretrained = pretrained['state_dict']
    if next(iter(pretrained)).startswith('module.'):
        pretrained = {k.replace('module.', ''): v for k, v in pretrained.items()}
    return pretrained


def _select_transferable_weights(pretrained, load_mode):
    if load_mode == 'all_compatible':
        return pretrained
    if load_mode != 'encoder_only':
        raise ValueError(f'Unknown PRETRAIN_LOAD_MODE: {load_mode}')
    return {
        k: v for k, v in pretrained.items()
        if k == 'absolute_pos_embed' or k.startswith(ENCODER_STATE_PREFIXES)
    }


def _load_pretrained_weights(net, pretrain_path, device, load_mode=PRETRAIN_LOAD_MODE):
    """Load pre-trained weights into the downstream SwinUnet.

    encoder_only: transfer patch embedding + encoder/bottleneck weights only.
    all_compatible: transfer every shape-compatible weight except the 2-class head.
    """
    pretrained = torch.load(pretrain_path, map_location=device, weights_only=False)
    pretrained = _normalize_pretrained_state_dict(pretrained)
    pretrained = _select_transferable_weights(pretrained, load_mode)

    model_dict = net.swin_unet.state_dict()
    compatible = {
        k: v for k, v in pretrained.items()
        if k in model_dict and v.shape == model_dict[k].shape
    }
    skipped = sorted(set(pretrained) - set(compatible))

    model_dict.update(compatible)
    net.swin_unet.load_state_dict(model_dict)

    print(
        f"  Loaded {len(compatible)}/{len(model_dict)} weights from {pretrain_path} "
        f"using mode={load_mode}"
    )
    if skipped:
        preview = ', '.join(skipped[:8])
        suffix = ' ...' if len(skipped) > 8 else ''
        print(f"  Skipped {len(skipped)} checkpoint keys: {preview}{suffix}")
    return net


def _train_fold(fold_id, output_dir, device, resume=None):
    """Train a single fold and return the trained model + config."""
    args = _make_args(fold_id, output_dir, resume=resume)
    config = get_config(args)
    os.makedirs(output_dir, exist_ok=True)

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
    if torch.cuda.is_available():
        cudnn.benchmark = True

    net = SwinUnet(config, img_size=config.DATA.IMG_SIZE, num_classes=2,
                   use_factored_embed=USE_FACTORED_EMBED).to(device)

    if PRETRAIN_WEIGHTS is not None:
        _load_pretrained_weights(net, PRETRAIN_WEIGHTS, device, load_mode=PRETRAIN_LOAD_MODE)

    trainer_wildfire(args, net, output_dir, device=device)
    return net, args


def _eval_fold(net, fold_id, device):
    """Evaluate a trained model on the test split and return metrics dict."""
    train_years, _, test_years = get_year_split(fold_id)

    db_test = WildfireDataset(
        data_dir=HDF5_DIR,
        included_fire_years=test_years,
        is_train=False,
        stats_years=train_years,
        n_leading_observations=N_LEADING_OBSERVATIONS,
        n_leading_observations_test_adjustment=N_LEADING_OBSERVATIONS,
        crop_side_length=CROP_SIDE_LENGTH,
        load_from_hdf5=True,
        use_factored_embed=USE_FACTORED_EMBED,
    )
    test_loader = DataLoader(
        db_test, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(str(device) == 'cuda'),
    )

    net.eval()
    all_probs, all_preds, all_gts = [], [], []
    with torch.no_grad():
        for x_batch, y_batch in tqdm(test_loader, desc=f"  Test fold {fold_id}", leave=False):
            x_batch = x_batch.to(device)
            logits = net(x_batch)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = (probs >= 0.5).astype(np.int64)
            gts = y_batch.numpy()
            all_probs.append(probs.flatten())
            all_preds.append(preds.flatten())
            all_gts.append(gts.flatten())

    all_probs = np.concatenate(all_probs)
    all_preds = np.concatenate(all_preds)
    all_gts = np.concatenate(all_gts)
    metrics = compute_binary_metrics(all_preds, all_gts)
    ap = compute_ap(all_probs, all_gts)
    return dict(ap=ap, **metrics)


def _load_best_weights(net, output_dir, device):
    """Load best_model.pth into net."""
    ckpt_path = os.path.join(output_dir, 'best_model.pth')
    state_dict = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(state_dict, dict) and 'model_state' in state_dict:
        state_dict = state_dict['model_state']
    if next(iter(state_dict)).startswith('module.'):
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    net.load_state_dict(state_dict)
    return net


print("Helpers defined. Run the next cell to start training.")

In [ ]:
folds_to_run = list(range(12)) if RUN_ALL_FOLDS else [DATA_FOLD_ID]
all_results = {}

embed_tag = "factored" if USE_FACTORED_EMBED else "stacked"
print(f"Patch embedding: {embed_tag}")
print(f"LR={BASE_LR}, epochs={MAX_EPOCHS}, batch={BATCH_SIZE}")
print(f"Folds to run: {folds_to_run}")
print("=" * 60)

for fold_id in folds_to_run:
    fold_dir = f"{OUTPUT_BASE}/fold{fold_id}"
    print(f"\n{'='*60}")
    print(f"FOLD {fold_id}/11  ->  {fold_dir}")
    print(f"{'='*60}")

    # Train
    net, args = _train_fold(fold_id, fold_dir, DEVICE,
                            resume=RESUME_CHECKPOINT if not RUN_ALL_FOLDS else None)

    # Load best weights and evaluate on test split
    net = _load_best_weights(net, fold_dir, DEVICE)
    metrics = _eval_fold(net, fold_id, DEVICE)
    all_results[fold_id] = metrics

    print(f"\nFold {fold_id} test:  AP={metrics['ap']:.4f}  "
          f"F1={metrics['f1']:.4f}  Prec={metrics['precision']:.4f}  "
          f"Rec={metrics['recall']:.4f}")

    # Free GPU memory between folds
    del net
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"{'Fold':>6}  {'AP':>8}  {'F1':>8}  {'Prec':>8}  {'Rec':>8}")
print("-" * 46)
for fid in sorted(all_results):
    m = all_results[fid]
    print(f"{fid:>6}  {m['ap']:>8.4f}  {m['f1']:>8.4f}  {m['precision']:>8.4f}  {m['recall']:>8.4f}")

if len(all_results) > 1:
    aps  = [m['ap']        for m in all_results.values()]
    f1s  = [m['f1']        for m in all_results.values()]
    precs = [m['precision'] for m in all_results.values()]
    recs  = [m['recall']   for m in all_results.values()]
    print("-" * 46)
    print(f"{'Mean':>6}  {np.mean(aps):>8.4f}  {np.mean(f1s):>8.4f}  "
          f"{np.mean(precs):>8.4f}  {np.mean(recs):>8.4f}")
    print(f"{'Std':>6}  {np.std(aps):>8.4f}  {np.std(f1s):>8.4f}  "
          f"{np.std(precs):>8.4f}  {np.std(recs):>8.4f}")
print("=" * 60)